In [ ]:
from PIL import Image
import numpy as np

for name in ['sheep', 'hamster', "cat", "bear"]:
    img = Image.open(f'/workspace/DL_final/datasets/characters/{name}.png').convert('RGBA')
    arr = np.array(img)
    # alpha 채널로 캐릭터 범위 파악
    alpha = arr[:,:,3]
    rows = np.where(alpha > 10)
    if len(rows[0]) > 0:
        y_min, y_max = rows[0].min(), rows[0].max()
        x_min, x_max = rows[1].min(), rows[1].max()
        print(f'{name}: size={img.size}, body bbox: y={y_min}-{y_max}, x={x_min}-{x_max}')

sheep: size=(736, 736), body bbox: y=0-735, x=0-735
hamster: size=(736, 736), body bbox: y=0-735, x=0-735
cat: size=(300, 300), body bbox: y=0-299, x=0-299
bear: size=(736, 736), body bbox: y=0-735, x=0-735


In [4]:
img = Image.open('/workspace/DL_final/datasets/characters/masking/hamster/hamster_top.png').convert('L')
arr = np.array(img)
whites = np.where(arr > 128)
if len(whites[0]) > 0:
    y_min, y_max = whites[0].min(), whites[0].max()
    x_min, x_max = whites[1].min(), whites[1].max()
    print(f'hamster_top: size={img.size}, white bbox: y={y_min}-{y_max}, x={x_min}-{x_max}')

hamster_top: size=(736, 736), white bbox: y=538-629, x=243-490


In [8]:

import shutil
from pathlib import Path

def extend_bottom_mask(mask_path: str, bottom_y: int) -> None:
    """각 x 컬럼에서 흰색 픽셀이 있으면 bottom_y까지 채움."""
    img = Image.open(mask_path).convert('L')
    arr = np.array(img, dtype=np.uint8)

    # 기존 흰색 픽셀이 있는 x 컬럼 파악
    for x in range(arr.shape[1]):
        col = arr[:, x]
        white_rows = np.where(col > 128)[0]
        if len(white_rows) > 0:
            y_top = white_rows.min()
            arr[y_top:bottom_y, x] = 255

    result = Image.fromarray(arr, mode='L')
    result.save(mask_path)
    print(f"저장: {mask_path}  (흰색 영역: y_top → {bottom_y})")

mask_dir = Path("/workspace/DL_final/datasets/characters/masking")

for name in ["cat"]:
    src = mask_dir / f"{name}" / f"{name}_bottom.png"
    shutil.copy(src, mask_dir / f"{name}_bottom_backup.png")
    print(f"백업: {name}_bottom_backup.png")

# 736 * 0.95 ≈ 699 → 발끝 포함
# 300 * 0.95 = 285
BOTTOM_Y = 285

extend_bottom_mask(str(mask_dir / "cat/cat_bottom.png"), BOTTOM_Y)
# extend_bottom_mask(str(mask_dir / "bear/bear_bottom.png"),   BOTTOM_Y)

백업: cat_bottom_backup.png
저장: /workspace/DL_final/datasets/characters/masking/cat/cat_bottom.png  (흰색 영역: y_top → 285)


In [2]:
from collections import deque
from pathlib import Path

import numpy as np
from PIL import Image, ImageFilter


def remove_outer_white_background(
    input_path: str | Path,
    output_path: str | Path,
    white_threshold: int = 235,
    feather_radius: int = 2,
):
    """
    이미지 외곽과 연결된 흰색 배경만 투명화.
    캐릭터 내부의 흰색 영역은 유지한다.
    """

    input_path = Path(input_path)
    output_path = Path(output_path)

    image = Image.open(input_path).convert("RGBA")
    arr = np.array(image)

    height, width = arr.shape[:2]
    rgb = arr[..., :3]

    # 흰색 후보 영역
    white_candidate = np.all(rgb >= white_threshold, axis=2)

    # 외곽과 연결된 흰색만 background로 판정
    background = np.zeros((height, width), dtype=bool)
    visited = np.zeros((height, width), dtype=bool)
    queue = deque()

    for x in range(width):
        queue.append((x, 0))
        queue.append((x, height - 1))

    for y in range(height):
        queue.append((0, y))
        queue.append((width - 1, y))

    while queue:
        x, y = queue.popleft()

        if visited[y, x]:
            continue

        visited[y, x] = True

        if not white_candidate[y, x]:
            continue

        background[y, x] = True

        for dx, dy in [
            (1, 0),
            (-1, 0),
            (0, 1),
            (0, -1),
        ]:
            nx = x + dx
            ny = y + dy

            if 0 <= nx < width and 0 <= ny < height:
                if not visited[ny, nx]:
                    queue.append((nx, ny))

    # 기본 alpha: 캐릭터 255, 외부 배경 0
    alpha = np.where(background, 0, 255).astype(np.uint8)

    # 경계만 부드럽게 처리
    if feather_radius > 0:
        alpha_image = Image.fromarray(alpha, mode="L")
        alpha_image = alpha_image.filter(
            ImageFilter.GaussianBlur(radius=feather_radius)
        )
        alpha = np.array(alpha_image)

        # 확실한 내부 영역은 다시 완전 불투명 처리
        alpha[~background & (alpha > 220)] = 255

    result = arr.copy()
    result[..., 3] = alpha

    output_path.parent.mkdir(parents=True, exist_ok=True)

    Image.fromarray(result, mode="RGBA").save(output_path)

    print(f"저장 완료: {output_path}")

In [3]:
input_dir = Path("/workspace/DL_final/datasets/characters/")
output_dir = Path("/workspace/DL_final/datasets/assets/characters_transparent")

extensions = {".png", ".jpg", ".jpeg", ".webp"}

for input_path in input_dir.rglob("*"):
    if not input_path.is_file():
        continue

    if input_path.suffix.lower() not in extensions:
        continue

    relative_path = input_path.relative_to(input_dir)

    output_path = (
        output_dir
        / relative_path.parent
        / f"{input_path.stem}_transparent.png"
    )

    try:
        remove_outer_white_background(
            input_path=input_path,
            output_path=output_path,
            white_threshold=235,
            feather_radius=2,
        )
    except Exception as e:
        print(f"실패: {input_path} / {e}")

저장 완료: /workspace/DL_final/datasets/assets/characters_transparent/hamster_transparent.png
저장 완료: /workspace/DL_final/datasets/assets/characters_transparent/cat_transparent.png
저장 완료: /workspace/DL_final/datasets/assets/characters_transparent/raccoon_transparent.png
저장 완료: /workspace/DL_final/datasets/assets/characters_transparent/bear_transparent.png
저장 완료: /workspace/DL_final/datasets/assets/characters_transparent/sheep_transparent.png
저장 완료: /workspace/DL_final/datasets/assets/characters_transparent/squirrel_transparent.png
저장 완료: /workspace/DL_final/datasets/assets/characters_transparent/masking/cat/cat_top_transparent.png
저장 완료: /workspace/DL_final/datasets/assets/characters_transparent/masking/cat/cat_all_transparent.png
저장 완료: /workspace/DL_final/datasets/assets/characters_transparent/masking/cat/cat_bottom_transparent.png
저장 완료: /workspace/DL_final/datasets/assets/characters_transparent/masking/cat/cat_bottom_backup_transparent.png
저장 완료: /workspace/DL_final/datasets/assets/char

In [6]:
input_dir = Path("/workspace/DL_final/datasets/outputs/web_9fc465e3")
output_dir = Path("/workspace/DL_final/datasets/outputs/web_9fc465e3")

for name in ["combo_000_lower"]:
  input = input_dir / f"{name}.png"
  output = output_dir / f"{name}_transparent.png"
  

  try:
      remove_outer_white_background(
          input_path=input,
          output_path=output,
          white_threshold=245,
          feather_radius=2,
      )
  except Exception as e:
      print(f"실패: {input_path} / {e}")    

저장 완료: /workspace/DL_final/datasets/outputs/web_9fc465e3/combo_000_lower_transparent.png
